# 🧠 InfraMind — Unsloth Training Pipeline
Meta x Scaler AI Hackathon Finale 

This notebook demonstrates how to train an Open-Source LLM (LLaMA-3) to excel at the **InfraMind Autonomous DevOps Benchmark** using [Unsloth](https://github.com/unslothai/unsloth) for 2x faster training and 70% less VRAM usage.

**Theme Focus**: Theme 3 (World Modeling) & Theme 2 (Super Long-Horizon Planning)
**Goal**: Show observable reward improvement in incident recovery.

In [ ]:
%%capture
# 1. Install Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Initialize Unsloth & LLaMA-3
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 3. Load InfraMind Trajectories Dataset
# We use the data exported from our generate_dataset.py script
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Ensure you uploaded inframind_sft.jsonl to Colab
try:
    with open("inframind_sft.jsonl", "r") as f:
        data = [json.loads(line) for line in f]
    dataset = Dataset.from_list(data)
except FileNotFoundError:
    print("Please upload inframind_sft.jsonl generated by scripts/generate_dataset.py")

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 4. Train with TRL SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 5. Plot Reward / Loss Curves
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
loss = [x["loss"] for x in log_history if "loss" in x]
steps = [x["step"] for x in log_history if "loss" in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, loss, label="Training Loss (correlates with reward improvement)", color="blue")
plt.title("InfraMind Model Fine-Tuning Convergence")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 6. Save LoRA Adapters
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Model saved successfully! Ready for inference in InfraMind.")